# Evaluation Metrics (Qwen judge output)

This notebook computes the requested four metrics from:

- Output: `../output/test_main_eval_stream_batch_Qwen__Qwen3-Next-80B-A3B-Instruct-FP8.jsonl`

- Input reference set: `../input/combined_dataset_with_reference_good_row_idx.json`

Metrics:

1. **Perplexity (mu ± std)**

   - Uses two embedding models:

     - `google/embeddinggemma-300m`

     - `perplexity-ai/pplx-embed-v1-0.6b`

   - Since embedding models do not provide token-level perplexity, this notebook reports an **embedding-based traditional perplexity proxy**.

   - For each row output text `t_r`, compare against all references `R`, compute `p_match(r)` for the true question reference, then `ppl_embed(r) = exp(-log p_match(r))`.

   - `single_answer` rows use the concatenated `raw_output_a` and `raw_output_b` as the row text.

   - Therefore, each eligible row gets its own perplexity value per model.

2. **Agreement (mu ± std)**

   - Agreement with human/golden winner (`winner`) after mapping judge outputs to `{model_a, model_b, tie}`.

3. **Consistency (mu ± std)**

   - Multi-run consistency for the same question under same setting.

   - Uses **1-flip consistency** across repeats:

     - `1 - (# winner flips between adjacent repeats / (n-1))`

4. **Error Rate (mu ± std)**

   - Fraction of rows with wrong output format (invalid winner token or invalid single scores).

The final summary is aggregated over condition cells:

`judge_type x prompt_variant x temperature`.


In [1]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


def resolve_path(candidates: list[str]) -> Path:
    for cand in candidates:
        p = Path(cand)
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f"Could not resolve any of: {candidates}")


OUTPUT_PATH = resolve_path([
    "output/test_main_eval_stream_batch_Qwen__Qwen3-Next-80B-A3B-Instruct-FP8.jsonl",
    "../output/test_main_eval_stream_batch_Qwen__Qwen3-Next-80B-A3B-Instruct-FP8.jsonl",
])

INPUT_PATH = resolve_path([
    "input/combined_dataset_with_reference_good_row_idx.json",
    "../input/combined_dataset_with_reference_good_row_idx.json",
])

print("Output:", OUTPUT_PATH)
print("Input :", INPUT_PATH)


def _last_assistant_text(conv) -> str:
    if isinstance(conv, list):
        for msg in reversed(conv):
            if isinstance(msg, dict) and str(msg.get("role", "")).lower() == "assistant":
                return str(msg.get("content", "")).strip()
    return ""


# Output data (JSONL)
df_out = pd.read_json(OUTPUT_PATH, lines=True)

# Input reference data is JSONL despite .json suffix
df_ref = pd.read_json(INPUT_PATH, lines=True)

df_ref = df_ref.copy()
df_ref["answer_a_text"] = df_ref["conversation_a"].map(_last_assistant_text)
df_ref["answer_b_text"] = df_ref["conversation_b"].map(_last_assistant_text)
df_ref["gold_winner"] = df_ref["winner"]

merge_cols = ["row_idx", "dataset", "gold_winner", "reference_answer", "answer_a_text", "answer_b_text"]
df = df_out.merge(
    df_ref[merge_cols],
    how="left",
    left_on="question_id",
    right_on="row_idx",
)

print("\nLoaded output rows:", len(df_out))
print("Loaded input rows :", len(df_ref))
print("Merged rows       :", len(df))
print("Questions covered :", df["question_id"].nunique())
print("Judge types       :", df["judge_type"].value_counts().to_dict())
print("Prompt variants   :", df["prompt_variant"].value_counts().to_dict())
print("Temperatures      :", sorted(df["temperature"].dropna().unique().tolist()))
print("Repeats           :", sorted(df["repeat_id"].dropna().unique().tolist())[:3], "...", sorted(df["repeat_id"].dropna().unique().tolist())[-3:])
print("Datasets          :", df["dataset"].value_counts(dropna=False).to_dict())

missing_ref = int(df["reference_answer"].isna().sum())
missing_gold = int(df["gold_winner"].isna().sum())
print("Missing reference_answer rows:", missing_ref)
print("Missing gold_winner rows     :", missing_gold)

if missing_ref > 0:
    print("Sample question_id with missing reference:")
    display(df.loc[df["reference_answer"].isna(), ["dataset", "question_id"]].drop_duplicates().head(10))

Output: /home/snt/projects_lujun/LLMJudgeTempCausal/output/test_main_eval_stream_batch_Qwen__Qwen3-Next-80B-A3B-Instruct-FP8.jsonl
Input : /home/snt/projects_lujun/LLMJudgeTempCausal/input/combined_dataset_with_reference_good_row_idx.json

Loaded output rows: 180000
Loaded input rows : 500
Merged rows       : 180000
Questions covered : 500
Judge types       : {'pairwise': 60000, 'single_answer': 60000, 'reference_guided': 60000}
Prompt variants   : {'baseline': 90000, 'cot': 90000}
Temperatures      : [0.01, 0.5, 1.0, 1.5, 2.0, 3.0]
Repeats           : [0, 1, 2] ... [7, 8, 9]
Datasets          : {'mt_bench_human_judgments': 126000, 'mmlu_pro': 54000}
Missing reference_answer rows: 360
Missing gold_winner rows     : 0
Sample question_id with missing reference:


,dataset,question_id
439,mmlu_pro,439


In [2]:
# ---- Normalize judge outputs and compute core metrics (except embedding perplexity) ----

VALID_PAIRWISE = {"A", "B", "C"}


def map_pairwise_to_gold_label(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    if s == "A":
        return "model_a"
    if s == "B":
        return "model_b"
    if s == "C":
        return "tie"
    return np.nan


# Ensure object dtype columns for mixed string/NaN values
if "predicted_winner" not in df.columns:
    df["predicted_winner"] = pd.Series(index=df.index, dtype="object")
if "format_error" not in df.columns:
    df["format_error"] = pd.Series(index=df.index, dtype="object")


# Pairwise/reference rows: token-based winner
pair_mask = df["judge_type"].isin(["pairwise", "reference_guided"])
df.loc[pair_mask, "winner_token"] = df.loc[pair_mask, "pairwise_winner"].astype(str).str.strip().str.upper()
df.loc[pair_mask, "format_error"] = ~df.loc[pair_mask, "winner_token"].isin(VALID_PAIRWISE)
df.loc[pair_mask, "predicted_winner"] = df.loc[pair_mask, "winner_token"].map(map_pairwise_to_gold_label)


# Single-answer rows: score-based winner
single_mask = df["judge_type"].eq("single_answer")
sa = pd.to_numeric(df.loc[single_mask, "score_a"], errors="coerce")
sb = pd.to_numeric(df.loc[single_mask, "score_b"], errors="coerce")
valid_single = sa.notna() & sb.notna() & sa.between(1, 10) & sb.between(1, 10)

single_pred = pd.Series(
    np.where(sa > sb, "model_a", np.where(sa < sb, "model_b", "tie")),
    index=df.index[single_mask],
    dtype="object",
)
single_pred.loc[~valid_single] = None

df.loc[single_mask, "predicted_winner"] = single_pred
df.loc[single_mask, "format_error"] = ~valid_single


# Safety fill for any unknown type
df["format_error"] = df["format_error"].fillna(True).astype(bool)


# Agreement with gold winner from input file
df["agreement"] = np.where(
    df["predicted_winner"].notna() & df["gold_winner"].notna(),
    (df["predicted_winner"] == df["gold_winner"]).astype(float),
    np.nan,
)


# 1-flip consistency helper

def one_flip_consistency(labels: list[str]) -> float:
    if len(labels) <= 1:
        return np.nan
    flips = sum(labels[i] != labels[i - 1] for i in range(1, len(labels)))
    return 1.0 - flips / (len(labels) - 1)


cons_rows = []
for (ds, jt, pv, temp, qid), g in df.groupby(
    ["dataset", "judge_type", "prompt_variant", "temperature", "question_id"],
    dropna=False,
):
    seq = g.sort_values("repeat_id")["predicted_winner"].dropna().astype(str).tolist()
    cons_rows.append(
        {
            "dataset": ds,
            "judge_type": jt,
            "prompt_variant": pv,
            "temperature": temp,
            "question_id": qid,
            "consistency_1flip": one_flip_consistency(seq),
        }
    )

df_cons = pd.DataFrame(cons_rows)


# Condition-level aggregates used for mu±std summary
cond_keys_with_repeat = ["dataset", "judge_type", "prompt_variant", "temperature", "repeat_id"]
cond_keys_no_repeat = ["dataset", "judge_type", "prompt_variant", "temperature"]

agreement_by_run = (
    df.groupby(cond_keys_with_repeat, dropna=False)["agreement"]
    .mean()
    .rename("agreement")
    .reset_index()
)

error_by_run = (
    df.groupby(cond_keys_with_repeat, dropna=False)["format_error"]
    .mean()
    .rename("error_rate")
    .reset_index()
)

consistency_by_cond = (
    df_cons.groupby(cond_keys_no_repeat, dropna=False)["consistency_1flip"]
    .mean()
    .rename("consistency")
    .reset_index()
)

print("Format-error overall:", round(df["format_error"].mean() * 100, 2), "%")
print("Agreement rows (non-null):", int(df["agreement"].notna().sum()))
print("Consistency rows (question-level):", int(df_cons["consistency_1flip"].notna().sum()))

print("\nAgreement by run (sample):")
display(agreement_by_run.head(8))

print("Error-rate by run (sample):")
display(error_by_run.head(8))

print("Consistency by condition (sample):")
display(consistency_by_cond.head(8))

Format-error overall: 8.19 %
Agreement rows (non-null): 165259
Consistency rows (question-level): 17828

Agreement by run (sample):


,dataset,judge_type,prompt_variant,temperature,repeat_id,agreement
0,mmlu_pro,pairwise,baseline,0.01,0,0.573333
1,mmlu_pro,pairwise,baseline,0.01,1,0.593333
2,mmlu_pro,pairwise,baseline,0.01,2,0.580000
3,mmlu_pro,pairwise,baseline,0.01,3,0.580000
4,mmlu_pro,pairwise,baseline,0.01,4,0.573333
5,mmlu_pro,pairwise,baseline,0.01,5,0.566667
6,mmlu_pro,pairwise,baseline,0.01,6,0.573333
7,mmlu_pro,pairwise,baseline,0.01,7,0.586667


Error-rate by run (sample):


,dataset,judge_type,prompt_variant,temperature,repeat_id,error_rate
0,mmlu_pro,pairwise,baseline,0.01,0,0.0
1,mmlu_pro,pairwise,baseline,0.01,1,0.0
2,mmlu_pro,pairwise,baseline,0.01,2,0.0
3,mmlu_pro,pairwise,baseline,0.01,3,0.0
4,mmlu_pro,pairwise,baseline,0.01,4,0.0
5,mmlu_pro,pairwise,baseline,0.01,5,0.0
6,mmlu_pro,pairwise,baseline,0.01,6,0.0
7,mmlu_pro,pairwise,baseline,0.01,7,0.0


Consistency by condition (sample):


,dataset,judge_type,prompt_variant,temperature,consistency
0,mmlu_pro,pairwise,baseline,0.01,0.970370
1,mmlu_pro,pairwise,baseline,0.50,0.956296
2,mmlu_pro,pairwise,baseline,1.00,0.928148
3,mmlu_pro,pairwise,baseline,1.50,0.887685
4,mmlu_pro,pairwise,baseline,2.00,0.835444
5,mmlu_pro,pairwise,baseline,3.00,0.681201
6,mmlu_pro,pairwise,cot,0.01,0.949630
7,mmlu_pro,pairwise,cot,0.50,0.920838


In [3]:
# ---- Test mode: keep only one question per (dataset, judge_type, prompt_variant, temperature) ----

TEST_ONE_QUESTION_PER_CONDITION = False

if "df_full_for_test" not in globals():
    df_full_for_test = df.copy()

base_df = df_full_for_test.copy()

if TEST_ONE_QUESTION_PER_CONDITION:
    cond_keys_no_repeat = ["dataset", "judge_type", "prompt_variant", "temperature"]
    cond_keys_with_repeat = ["dataset", "judge_type", "prompt_variant", "temperature", "repeat_id"]

    picked_questions = (
        base_df[cond_keys_no_repeat + ["question_id"]]
        .drop_duplicates()
        .sort_values(cond_keys_no_repeat + ["question_id"])
        .groupby(cond_keys_no_repeat, as_index=False)
        .first()
    )

    df = base_df.merge(
        picked_questions,
        on=cond_keys_no_repeat + ["question_id"],
        how="inner",
    ).copy()

    # Recompute consistency and run-level aggregates from sampled df
    cons_rows = []
    for (ds, jt, pv, temp, qid), g in df.groupby(
        ["dataset", "judge_type", "prompt_variant", "temperature", "question_id"],
        dropna=False,
    ):
        seq = g.sort_values("repeat_id")["predicted_winner"].dropna().astype(str).tolist()
        cons_rows.append(
            {
                "dataset": ds,
                "judge_type": jt,
                "prompt_variant": pv,
                "temperature": temp,
                "question_id": qid,
                "consistency_1flip": one_flip_consistency(seq),
            }
        )

    df_cons = pd.DataFrame(cons_rows)

    agreement_by_run = (
        df.groupby(cond_keys_with_repeat, dropna=False)["agreement"]
        .mean()
        .rename("agreement")
        .reset_index()
    )

    error_by_run = (
        df.groupby(cond_keys_with_repeat, dropna=False)["format_error"]
        .mean()
        .rename("error_rate")
        .reset_index()
    )

    consistency_by_cond = (
        df_cons.groupby(cond_keys_no_repeat, dropna=False)["consistency_1flip"]
        .mean()
        .rename("consistency")
        .reset_index()
    )

    print("[TEST MODE] One question kept per condition.")
    print("Picked conditions:", len(picked_questions))
    print("Sampled rows      :", len(df))
    print("Sampled questions :", df["question_id"].nunique())
    print("Dataset counts    :", df["dataset"].value_counts().to_dict())
    print("Prompt counts     :", df["prompt_variant"].value_counts().to_dict())
    print("Judge type counts :", df["judge_type"].value_counts().to_dict())
else:
    df = base_df.copy()
    print("[TEST MODE OFF] Using full dataset rows:", len(df))

[TEST MODE OFF] Using full dataset rows: 180000


In [ ]:
# ---- Traditional embedding-based perplexity with two embedding models ----

MODEL_NAME_GEMMA = "google/embeddinggemma-300m"

MODEL_NAME_PPLX = "perplexity-ai/pplx-embed-v1-0.6b"

TAU = 0.07

BATCH_SIZE = 64

PPL_BATCH_SIZE = 512

def _l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:

    n = np.linalg.norm(x, axis=1, keepdims=True)

    return x / np.clip(n, eps, None)

def load_embedder(model_name: str):

    try:

        from sentence_transformers import SentenceTransformer

        st_model = SentenceTransformer(model_name)

        def encode_texts(texts: list[str]) -> np.ndarray:

            emb = st_model.encode(

                texts,

                batch_size=BATCH_SIZE,

                convert_to_numpy=True,

                normalize_embeddings=True,

                show_progress_bar=True,

            )

            return emb.astype(np.float32)

        return encode_texts, "sentence_transformers"

    except Exception as st_err:

        print(f"sentence_transformers path failed for {model_name}:", repr(st_err))

        print("Falling back to transformers AutoModel mean pooling.")

        import torch

        from transformers import AutoModel, AutoTokenizer

        device = "cuda" if torch.cuda.is_available() else "cpu"

        tok = AutoTokenizer.from_pretrained(model_name)

        mdl = AutoModel.from_pretrained(model_name).to(device)

        mdl.eval()

        def encode_texts(texts: list[str]) -> np.ndarray:

            out = []

            for i in tqdm(range(0, len(texts), BATCH_SIZE), desc=f"Embedding batches [{model_name}]"):

                batch = texts[i : i + BATCH_SIZE]

                enc = tok(

                    batch,

                    padding=True,

                    truncation=True,

                    max_length=2048,

                    return_tensors="pt",

                )

                enc = {k: v.to(device) for k, v in enc.items()}

                with torch.no_grad():

                    res = mdl(**enc)

                    if hasattr(res, "last_hidden_state"):

                        hs = res.last_hidden_state

                        mask = enc["attention_mask"].unsqueeze(-1)

                        pooled = (hs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

                    elif hasattr(res, "pooler_output") and res.pooler_output is not None:

                        pooled = res.pooler_output

                    else:

                        pooled = res[0][:, 0]

                out.append(pooled.detach().cpu().numpy())

            emb = np.concatenate(out, axis=0).astype(np.float32)

            return _l2_normalize(emb)

        return encode_texts, f"transformers({device})"

def probs_of_match(logits: np.ndarray, ref_idx: np.ndarray) -> np.ndarray:

    z = logits - logits.max(axis=1, keepdims=True)

    expz = np.exp(z)

    denom = np.clip(expz.sum(axis=1), 1e-12, None)

    p = expz[np.arange(len(ref_idx)), ref_idx] / denom

    return np.clip(p, 1e-12, 1.0)

qref = (

    df[["dataset", "question_id", "reference_answer"]]

    .drop_duplicates(["dataset", "question_id"])

    .sort_values(["dataset", "question_id"])

    .reset_index(drop=True)

)

qref["reference_answer"] = qref["reference_answer"].fillna("").astype(str).str.strip()

qref_valid = qref[qref["reference_answer"].ne("")].copy().reset_index(drop=True)

print("Questions with valid reference:", len(qref_valid), "/", len(qref))

ref_texts = qref_valid["reference_answer"].tolist()

ref_keys = list(qref_valid[["dataset", "question_id"]].itertuples(index=False, name=None))

key_to_ref_idx = {k: i for i, k in enumerate(ref_keys)}

row_keys = list(df[["dataset", "question_id"]].itertuples(index=False, name=None))

row_ref_idx = np.array([key_to_ref_idx.get(k, -1) for k in row_keys], dtype=np.int32)

candidate_cols = [c for c in ["raw_output", "judge_reason", "raw_output_a", "raw_output_b"] if c in df.columns]

if len(candidate_cols) == 0:

    raise KeyError("No raw judge output column found. Expected one of: raw_output, judge_reason, raw_output_a, raw_output_b")

raw_text = pd.Series("", index=df.index, dtype="object")

single_mask = df["judge_type"].eq("single_answer")

non_single_mask = ~single_mask

for c in candidate_cols:

    cur = df[c].fillna("").astype(str).str.strip()

    take = non_single_mask & raw_text.eq("") & cur.ne("")

    raw_text.loc[take] = cur.loc[take]

raw_output_a = (

    df["raw_output_a"].fillna("").astype(str).str.strip()

    if "raw_output_a" in df.columns

    else pd.Series("", index=df.index, dtype="object")

)

raw_output_b = (

    df["raw_output_b"].fillna("").astype(str).str.strip()

    if "raw_output_b" in df.columns

    else pd.Series("", index=df.index, dtype="object")

)

combined_single = pd.Series("", index=df.index, dtype="object")

both_mask = raw_output_a.ne("") & raw_output_b.ne("")

only_a_mask = raw_output_a.ne("") & raw_output_b.eq("")

only_b_mask = raw_output_a.eq("") & raw_output_b.ne("")

combined_single.loc[both_mask] = (

    "raw_output_a:\n" + raw_output_a.loc[both_mask] + "\n\nraw_output_b:\n" + raw_output_b.loc[both_mask]

)

combined_single.loc[only_a_mask] = "raw_output_a:\n" + raw_output_a.loc[only_a_mask]

combined_single.loc[only_b_mask] = "raw_output_b:\n" + raw_output_b.loc[only_b_mask]

raw_text.loc[single_mask] = combined_single.loc[single_mask]

single_empty_mask = single_mask & raw_text.eq("")

for c in ["raw_output", "judge_reason"]:

    if c in df.columns:

        cur = df[c].fillna("").astype(str).str.strip()

        take = single_empty_mask & cur.ne("")

        raw_text.loc[take] = cur.loc[take]

        single_empty_mask = single_mask & raw_text.eq("")

df["raw_judge_text"] = raw_text.replace("", np.nan)

for col in ["baseline_lookup_answer_text", "ppl_embed_traditional"]:

    if col in df.columns:

        df.drop(columns=col, inplace=True)

def compute_traditional_ppl_for_model(model_name: str, ppl_col: str) -> tuple[int, str]:

    encode_texts, backend = load_embedder(model_name)

    print(f"Embedding backend [{model_name}] :", backend)

    ref_matrix = encode_texts(ref_texts).astype(np.float32)

    ref_matrix = _l2_normalize(ref_matrix)

    ppl_values = np.full(len(df), np.nan, dtype=np.float32)

    valid_mask = df["raw_judge_text"].notna().to_numpy() & (row_ref_idx >= 0)

    valid_indices = np.flatnonzero(valid_mask)

    for start in tqdm(range(0, len(valid_indices), PPL_BATCH_SIZE), desc=f"Traditional ppl [{model_name}]"):

        idx = valid_indices[start : start + PPL_BATCH_SIZE]

        texts = df.iloc[idx]["raw_judge_text"].astype(str).tolist()

        emb = encode_texts(texts).astype(np.float32)

        emb = _l2_normalize(emb)

        logits = (emb @ ref_matrix.T) / TAU

        probs = probs_of_match(logits, row_ref_idx[idx])

        ppl_values[idx] = np.exp(-np.log(probs))

    df[ppl_col] = ppl_values.astype(float)

    return int(df[ppl_col].notna().sum()), backend

rows_gemma, backend_gemma = compute_traditional_ppl_for_model(MODEL_NAME_GEMMA, "ppl_embed")

rows_pplx, backend_pplx = compute_traditional_ppl_for_model(MODEL_NAME_PPLX, "ppl_embed_pplx")

df["ppl_method"] = pd.Series(np.nan, index=df.index, dtype="object")

df.loc[df["ppl_embed"].notna(), "ppl_method"] = "traditional_raw_output"

ppl_lookup = df[[

    "dataset", "question_id", "repeat_id", "judge_type", "prompt_variant", "temperature",

    "predicted_winner", "ppl_method", "raw_judge_text", "ppl_embed", "ppl_embed_pplx"

]].rename(columns={

    "ppl_embed": "perplexity_gemma",

    "ppl_embed_pplx": "perplexity_pplx",

}).copy()

single_rows = int(single_mask.sum())

single_with_text = int((single_mask & df["raw_judge_text"].notna()).sum())

print("Raw text source priority:", candidate_cols)

print("Rows with non-empty raw judge text:", int(df["raw_judge_text"].notna().sum()), "/", len(df))

print("Single-answer rows with raw_judge_text:", single_with_text, "/", single_rows)

print("Perplexity rows (gemma, non-null):", rows_gemma)

print("Perplexity rows (pplx, non-null) :", rows_pplx)

print("Rows missing reference map:", int((row_ref_idx < 0).sum()))

coverage_by_prompt = (

    df.assign(

        has_ppl_gemma=df["ppl_embed"].notna(),

        has_ppl_pplx=df["ppl_embed_pplx"].notna(),

    )

    .groupby("prompt_variant", dropna=False)[["has_ppl_gemma", "has_ppl_pplx"]]

    .mean()

    .mul(100)

    .rename(columns={"has_ppl_gemma": "coverage_gemma_percent", "has_ppl_pplx": "coverage_pplx_percent"})

    .reset_index()

)

print("Coverage by prompt_variant (%):")

display(coverage_by_prompt)

method_counts = df["ppl_method"].value_counts(dropna=False).rename_axis("ppl_method").to_frame("rows")

print("Perplexity route counts:")

display(method_counts)

Questions with valid reference: 499 / 500
Embedding backend [google/embeddinggemma-300m] : sentence_transformers


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Traditional ppl [google/embeddinggemma-300m]:   0%|          | 0/351 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

...

Embedding batches [perplexity-ai/pplx-embed-v1-0.6b]:   0%|          | 0/8 [00:00<?, ?it/s]

Embedding batches [perplexity-ai/pplx-embed-v1-0.6b]:   0%|          | 0/8 [00:00<?, ?it/s]

Embedding batches [perplexity-ai/pplx-embed-v1-0.6b]:   0%|          | 0/8 [00:00<?, ?it/s]

Embedding batches [perplexity-ai/pplx-embed-v1-0.6b]:   0%|          | 0/8 [00:00<?, ?it/s]

Embedding batches [perplexity-ai/pplx-embed-v1-0.6b]:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
df.to_jsonl("output/evaluation_with_ppl_mass.jsonl", orient="records", lines=True)

In [ ]:
if "ppl_embed_pplx" not in df.columns:

    df["ppl_embed_pplx"] = np.nan

perplexity_by_run = (

    df.groupby(["dataset", "judge_type", "prompt_variant", "temperature", "repeat_id"], dropna=False)

    .agg(

        perplexity_gemma=("ppl_embed", "mean"),

        perplexity_pplx=("ppl_embed_pplx", "mean"),

    )

    .reset_index()

)

coverage_gemma = float(df["ppl_embed"].notna().mean() * 100)

coverage_pplx = float(df["ppl_embed_pplx"].notna().mean() * 100)

print("Perplexity rows (gemma, non-null):", int(df["ppl_embed"].notna().sum()))

print("Perplexity coverage (gemma)      :", f"{coverage_gemma:.2f}%")

print("Perplexity rows (pplx, non-null) :", int(df["ppl_embed_pplx"].notna().sum()))

print("Perplexity coverage (pplx)       :", f"{coverage_pplx:.2f}%")

print("Perplexity by run sample:")

display(perplexity_by_run)

print("Per-row perplexity sample:")

display(ppl_lookup.head(8))

In [ ]:
# ---- Final summary: mu ± std (overall + by dataset) ----

def mu_std(series: pd.Series) -> tuple[float, float]:

    s = pd.to_numeric(series, errors="coerce").dropna()

    return float(s.mean()), float(s.std(ddof=1))

def metric_rows(

    scope: str,

    dataset_label: str,

    p_gemma_series: pd.Series,

    p_pplx_series: pd.Series,

    a_series: pd.Series,

    c_series: pd.Series,

    e_series: pd.Series,

) -> list[dict]:

    p_gemma_mu, p_gemma_std = mu_std(p_gemma_series)

    p_pplx_mu, p_pplx_std = mu_std(p_pplx_series)

    a_mu, a_std = mu_std(a_series)

    c_mu, c_std = mu_std(c_series)

    e_mu, e_std = mu_std(e_series)

    return [

        {

            "scope": scope,

            "dataset": dataset_label,

            "Metric": "Perplexity (embeddinggemma)",

            "mu": p_gemma_mu,

            "std": p_gemma_std,

            "mu±std": f"{p_gemma_mu:.4f} ± {p_gemma_std:.4f}",

            "mu%±std%": np.nan,

        },

        {

            "scope": scope,

            "dataset": dataset_label,

            "Metric": "Perplexity (pplx-embed)",

            "mu": p_pplx_mu,

            "std": p_pplx_std,

            "mu±std": f"{p_pplx_mu:.4f} ± {p_pplx_std:.4f}",

            "mu%±std%": np.nan,

        },

        {

            "scope": scope,

            "dataset": dataset_label,

            "Metric": "Agreement",

            "mu": a_mu,

            "std": a_std,

            "mu±std": f"{a_mu:.4f} ± {a_std:.4f}",

            "mu%±std%": f"{a_mu*100:.2f}% ± {a_std*100:.2f}%",

        },

        {

            "scope": scope,

            "dataset": dataset_label,

            "Metric": "Consistency (1-flip)",

            "mu": c_mu,

            "std": c_std,

            "mu±std": f"{c_mu:.4f} ± {c_std:.4f}",

            "mu%±std%": f"{c_mu*100:.2f}% ± {c_std*100:.2f}%",

        },

        {

            "scope": scope,

            "dataset": dataset_label,

            "Metric": "Error Rate",

            "mu": e_mu,

            "std": e_std,

            "mu±std": f"{e_mu:.4f} ± {e_std:.4f}",

            "mu%±std%": f"{e_mu*100:.2f}% ± {e_std*100:.2f}%",

        },

    ]

summary_rows = []

summary_rows += metric_rows(

    scope="overall",

    dataset_label="ALL",

    p_gemma_series=perplexity_by_run["perplexity_gemma"],

    p_pplx_series=perplexity_by_run["perplexity_pplx"],

    a_series=agreement_by_run["agreement"],

    c_series=consistency_by_cond["consistency"],

    e_series=error_by_run["error_rate"],

)

for ds in sorted(df["dataset"].dropna().astype(str).unique().tolist()):

    summary_rows += metric_rows(

        scope="by_dataset",

        dataset_label=ds,

        p_gemma_series=perplexity_by_run.loc[perplexity_by_run["dataset"] == ds, "perplexity_gemma"],

        p_pplx_series=perplexity_by_run.loc[perplexity_by_run["dataset"] == ds, "perplexity_pplx"],

        a_series=agreement_by_run.loc[agreement_by_run["dataset"] == ds, "agreement"],

        c_series=consistency_by_cond.loc[consistency_by_cond["dataset"] == ds, "consistency"],

        e_series=error_by_run.loc[error_by_run["dataset"] == ds, "error_rate"],

    )

summary = pd.DataFrame(summary_rows)

print("Requested metrics (mu ± std):")

display(summary[["scope", "dataset", "Metric", "mu±std", "mu%±std%"]])

group_cols = ["dataset", "judge_type", "prompt_variant", "temperature"]

agreement_by_cond = (

    agreement_by_run.groupby(group_cols, dropna=False)["agreement"]

    .mean()

    .reset_index()

)

error_by_cond = (

    error_by_run.groupby(group_cols, dropna=False)["error_rate"]

    .mean()

    .reset_index()

)

perplexity_gemma_by_cond = (

    perplexity_by_run.groupby(group_cols, dropna=False)["perplexity_gemma"]

    .mean()

    .reset_index()

)

perplexity_pplx_by_cond = (

    perplexity_by_run.groupby(group_cols, dropna=False)["perplexity_pplx"]

    .mean()

    .reset_index()

)

cond_table = (

    agreement_by_cond.merge(error_by_cond, on=group_cols, how="outer")

    .merge(consistency_by_cond, on=group_cols, how="outer")

    .merge(perplexity_gemma_by_cond, on=group_cols, how="outer")

    .merge(perplexity_pplx_by_cond, on=group_cols, how="outer")

    .sort_values(["dataset", "judge_type", "prompt_variant", "temperature"])

    .reset_index(drop=True)

)

print("\nCondition breakdown (by dataset):")

display(cond_table)

summary_path = OUTPUT_PATH.with_name("evaluation_summary_mu_std.csv")

summary_by_dataset_path = OUTPUT_PATH.with_name("evaluation_summary_mu_std_by_dataset.csv")

cond_path = OUTPUT_PATH.with_name("evaluation_by_condition.csv")

summary.to_csv(summary_path, index=False)

summary.to_csv(summary_by_dataset_path, index=False)

cond_table.to_csv(cond_path, index=False)

print("\nSaved:")

print("-", summary_path)

print("-", summary_by_dataset_path)

print("-", cond_path)

In [ ]:
# ---- Table like manuscript format: Human Bench vs MMLU_PRO ----

import numpy as np

import pandas as pd

def _fmt_mu_std(series: pd.Series, digits: int = 4) -> str:

    s = pd.to_numeric(series, errors="coerce").dropna()

    if len(s) == 0:

        return np.nan

    mu = float(s.mean())

    std = float(s.std(ddof=1)) if len(s) > 1 else 0.0

    return f"{mu:.{digits}f} ± {std:.{digits}f}"

def _short_model_name(x: str) -> str:

    if pd.isna(x):

        return x

    s = str(x)

    s = s.split("/")[-1]

    s = s.replace("-A3B-Instruct-FP8", "")

    s = s.replace("-Instruct", "")

    return s

def _one_flip_consistency(labels: list[str]) -> float:

    if len(labels) <= 1:

        return np.nan

    flips = sum(labels[i] != labels[i - 1] for i in range(1, len(labels)))

    return 1.0 - flips / (len(labels) - 1)

judge_type_map = {

    "pairwise": "Pairwise Comparison",

    "single_answer": "Single Answer Grading",

    "reference_guided": "Reference Guided Grading",

}

prompt_map = {"baseline": "Direct", "cot": "COT"}

run_group_cols = ["dataset", "judge_model", "judge_type", "prompt_variant", "temperature", "repeat_id"]

agreement_run_model = (

    df.groupby(run_group_cols, dropna=False)["agreement"]

    .mean()

    .rename("agreement")

    .reset_index()

)

error_run_model = (

    df.groupby(run_group_cols, dropna=False)["format_error"]

    .mean()

    .rename("error_rate")

    .reset_index()

)

perplexity_gemma_run_model = (

    df.groupby(run_group_cols, dropna=False)["ppl_embed"]

    .mean()

    .rename("perplexity_gemma")

    .reset_index()

)

perplexity_pplx_run_model = (

    df.groupby(run_group_cols, dropna=False)["ppl_embed_pplx"]

    .mean()

    .rename("perplexity_pplx")

    .reset_index()

)

cons_records = []

for key, g in df.groupby(["dataset", "judge_model", "judge_type", "prompt_variant", "temperature", "question_id"], dropna=False):

    seq = g.sort_values("repeat_id")["predicted_winner"].dropna().astype(str).tolist()

    cons_records.append(

        {

            "dataset": key[0],

            "judge_model": key[1],

            "judge_type": key[2],

            "prompt_variant": key[3],

            "temperature": key[4],

            "question_id": key[5],

            "consistency_1flip": _one_flip_consistency(seq),

        }

    )

cons_q_model = pd.DataFrame(cons_records)

cond_cols = ["dataset", "judge_model", "judge_type", "prompt_variant", "temperature"]

cond_keys = (

    df[cond_cols]

    .drop_duplicates()

    .sort_values(["dataset", "judge_model", "judge_type", "prompt_variant", "temperature"])

    .reset_index(drop=True)

)

rows = []

for r in cond_keys.itertuples(index=False):

    mask = (

        (agreement_run_model["dataset"] == r.dataset)

        & (agreement_run_model["judge_model"] == r.judge_model)

        & (agreement_run_model["judge_type"] == r.judge_type)

        & (agreement_run_model["prompt_variant"] == r.prompt_variant)

        & (agreement_run_model["temperature"] == r.temperature)

    )

    a_series = agreement_run_model.loc[mask, "agreement"]

    e_series = error_run_model.loc[mask, "error_rate"]

    p_gemma_series = perplexity_gemma_run_model.loc[mask, "perplexity_gemma"]

    p_pplx_series = perplexity_pplx_run_model.loc[mask, "perplexity_pplx"]

    c_mask = (

        (cons_q_model["dataset"] == r.dataset)

        & (cons_q_model["judge_model"] == r.judge_model)

        & (cons_q_model["judge_type"] == r.judge_type)

        & (cons_q_model["prompt_variant"] == r.prompt_variant)

        & (cons_q_model["temperature"] == r.temperature)

    )

    c_series = cons_q_model.loc[c_mask, "consistency_1flip"]

    rows.append(

        {

            "dataset": r.dataset,

            "T": r.temperature,

            "Model": _short_model_name(r.judge_model),

            "Judge Type": judge_type_map.get(r.judge_type, r.judge_type),

            "Prompt Strategy": prompt_map.get(r.prompt_variant, r.prompt_variant),

            "Perplexity Gemma (µ ± std)": _fmt_mu_std(p_gemma_series, digits=2),

            "Perplexity PPLX (µ ± std)": _fmt_mu_std(p_pplx_series, digits=2),

            "Aggrement (µ ± std)": _fmt_mu_std(a_series, digits=2),

            "Consistency (µ ± std)": _fmt_mu_std(c_series, digits=2),

            "Error Rate (µ ± std)": _fmt_mu_std(e_series, digits=2),

        }

    )

stats_long = pd.DataFrame(rows)

datasets = stats_long["dataset"].dropna().astype(str).unique().tolist()

human_ds = next((d for d in datasets if "mt_bench" in d.lower() or "human" in d.lower()), datasets[0] if datasets else None)

mmlu_ds = next((d for d in datasets if "mmlu" in d.lower()), datasets[1] if len(datasets) > 1 else None)

if human_ds is None or mmlu_ds is None:

    raise ValueError(f"Need two datasets to build this table. Found: {datasets}")

base_cols = ["T", "Model", "Judge Type", "Prompt Strategy"]

metric_cols = [

    "Perplexity Gemma (µ ± std)",

    "Perplexity PPLX (µ ± std)",

    "Aggrement (µ ± std)",

    "Consistency (µ ± std)",

    "Error Rate (µ ± std)",

]

human_part = stats_long.loc[stats_long["dataset"] == human_ds, base_cols + metric_cols].copy()

human_part = human_part.rename(columns={c: f"Human Bench {c}" for c in metric_cols})

mmlu_part = stats_long.loc[stats_long["dataset"] == mmlu_ds, base_cols + metric_cols].copy()

mmlu_part = mmlu_part.rename(columns={c: f"MMLU_PRO {c}" for c in metric_cols})

paper_table = (

    human_part.merge(mmlu_part, on=base_cols, how="outer")

    .sort_values(["T", "Model", "Judge Type", "Prompt Strategy"])

    .reset_index(drop=True)

)

print(f"Human Bench dataset: {human_ds}")

print(f"MMLU_PRO dataset: {mmlu_ds}")

display(paper_table)

paper_table_path = OUTPUT_PATH.with_name("evaluation_paper_style_table.csv")

paper_table.to_csv(paper_table_path, index=False)

print("Saved:", paper_table_path)